# Feature Shift: What It Is, How to Measure It, What It Means
## Applied to five grocery store datasets

**Feature shift** (also called *covariate shift*) occurs when the distribution
of input features differs between training data (source) and new data (target),
even if the underlying relationship between features and labels is the same.

This notebook walks through six quantification methods in detail:

| # | Method | Type | What it detects |
|---|---|---|---|
| 1 | KS statistic | Per-feature | Max CDF gap — any shape difference |
| 2 | Wasserstein distance | Per-feature | Mean shift + spread change |
| 3 | Mean shift & std ratio | Per-feature | First two moments |
| 4 | Jensen-Shannon divergence | Per-feature | Full distributional shape |
| 5 | Classifier discrepancy | Joint (all features) | Including interactions |
| 6 | PCA centroid distance | Joint (all features) | Overall multivariate shift |

**Source:** Kroger + Safeway + Walmart (median tier)  
**Targets:** Whole Foods (high tier) and Thrift Store (low tier)

## 1. Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from scipy.stats import ks_2samp, wasserstein_distance, entropy, skew
from scipy.stats import spearmanr
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# ── Store and feature config ──────────────────────────────────────────────────
SOURCE_STORES  = ["kroger", "safeway", "walmart"]
TARGET_STORES  = ["whole_foods", "thrift_store"]
ALL_STORES     = ["whole_foods","kroger","safeway","walmart","thrift_store"]
STORE_LABELS   = {"whole_foods":"Whole Foods","kroger":"Kroger",
                  "safeway":"Safeway","walmart":"Walmart","thrift_store":"Thrift"}
STORE_COLORS   = {"whole_foods":"#185FA5","kroger":"#1D9E75","safeway":"#0F6E56",
                  "walmart":"#854F0B","thrift_store":"#A32D2D"}
SOURCE_COLOR   = "#888780"

FEAT_COLS = ["age","visits_per_month","avg_basket_usd","monthly_spend_usd",
             "grocery_pct","electronics_pct","apparel_pct","home_pct",
             "private_label_pct","online_orders_pct","coupon_usage_pct",
             "loyalty_score","segment_enc"]
FEAT_NICE = ["Age","Visits/mo","Basket $","Monthly spend $","Grocery %",
             "Electronics %","Apparel %","Home %","Private label %",
             "Online orders %","Coupon use %","Loyalty score","Segment"]

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.5,
    "font.size": 10,
})
print("Setup complete.")

## 2. Load data

In [ ]:
df = pd.read_csv("grocery_all_stores.csv")
le = LabelEncoder()
le.fit(df["segment"])
df["segment_enc"] = le.transform(df["segment"])

scaler = StandardScaler()
scaler.fit(df[FEAT_COLS].values.astype(float))

source_df = df[df["store"].isin(SOURCE_STORES)].copy()
wf_df     = df[df["store"] == "whole_foods"].copy()
th_df     = df[df["store"] == "thrift_store"].copy()

X_src = scaler.transform(source_df[FEAT_COLS].values.astype(float))
X_wf  = scaler.transform(wf_df[FEAT_COLS].values.astype(float))
X_th  = scaler.transform(th_df[FEAT_COLS].values.astype(float))

print(f"Source (Kroger+Safeway+Walmart): {len(X_src)} rows")
print(f"Target — Whole Foods:            {len(X_wf)} rows")
print(f"Target — Thrift Store:           {len(X_th)} rows")

## 3. What is feature shift? — visual intuition

Before measuring anything, let's see what feature shift *looks like*.
We plot the distribution of three key features side by side for all three domains.

**No shift** would mean all three density curves overlap completely.
**Large shift** means the curves are in completely different positions.

In [ ]:
KEY3 = ["avg_basket_usd", "coupon_usage_pct", "private_label_pct"]
KEY3_NICE = ["Avg basket ($)", "Coupon use (%)", "Private label (%)"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Feature distributions: source vs Whole Foods vs Thrift Store",
             fontsize=12, fontweight="bold")

groups = [
    ("Source (Kroger+Safeway+Walmart)", source_df, SOURCE_COLOR, "-"),
    ("Whole Foods",                     wf_df,     STORE_COLORS["whole_foods"], "--"),
    ("Thrift Store",                    th_df,     STORE_COLORS["thrift_store"], ":"),
]

for ax, feat, nice in zip(axes, KEY3, KEY3_NICE):
    for label, sub_df, col, ls in groups:
        vals = sub_df[feat].values
        ax.hist(vals, bins=25, density=True, alpha=0.35, color=col)
        # Smooth KDE overlay
        from scipy.stats import gaussian_kde
        kde  = gaussian_kde(vals, bw_method=0.3)
        xs   = np.linspace(vals.min(), vals.max(), 200)
        ax.plot(xs, kde(xs), color=col, linewidth=2, linestyle=ls, label=label)
    ax.set_title(nice, fontweight="bold")
    ax.set_xlabel("Value"); ax.set_ylabel("Density")

axes[0].legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

print("Observation:")
print("  avg_basket_usd:   three clearly separate distributions -> large shift")
print("  coupon_usage_pct: Thrift far right, WF far left -> opposite shift directions")
print("  private_label_pct: similar pattern to coupon — both income proxies")

## 4. Method 1 — KS statistic (Kolmogorov-Smirnov)

**Formula:** KS = max_x |F_source(x) − F_target(x)|

Where F is the empirical CDF. The KS statistic is the maximum vertical
gap between the two CDF curves at any point x.

**Range:** 0 (identical CDFs) to 1 (completely non-overlapping)  
**Detects:** Any shape difference — bimodality, skew, tail differences  
**Misses:** Uniform shifts where both CDFs slide together  
**Threshold:** <0.2 LOW, 0.2–0.4 MED, >0.4 HIGH

In [ ]:
# Compute KS for every feature, both targets
ks_wf = [ks_2samp(X_src[:,i], X_wf[:,i]) for i in range(len(FEAT_COLS))]
ks_th = [ks_2samp(X_src[:,i], X_th[:,i]) for i in range(len(FEAT_COLS))]

ks_stat_wf = [r.statistic for r in ks_wf]
ks_stat_th = [r.statistic for r in ks_th]
ks_pval_wf = [r.pvalue    for r in ks_wf]
ks_pval_th = [r.pvalue    for r in ks_th]

# ── Bar chart of KS statistics ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle("Method 1 — KS statistics: source vs each target", fontweight="bold")

for ax, ks_vals, target_label, col in [
    (axes[0], ks_stat_wf, "Whole Foods",  STORE_COLORS["whole_foods"]),
    (axes[1], ks_stat_th, "Thrift Store", STORE_COLORS["thrift_store"]),
]:
    sorted_idx = np.argsort(ks_vals)[::-1]
    bar_colors = [col if v > 0.4 else ("#EF9F27" if v > 0.2 else "#B4B2A9")
                  for v in [ks_vals[i] for i in sorted_idx]]
    bars = ax.barh([FEAT_NICE[i] for i in sorted_idx],
                   [ks_vals[i]   for i in sorted_idx],
                   color=bar_colors, edgecolor="white", height=0.65)
    ax.axvline(0.4, color="crimson", linestyle="--", linewidth=1.2,
               alpha=0.7, label="HIGH (>0.4)")
    ax.axvline(0.2, color="darkorange", linestyle="--", linewidth=1.0,
               alpha=0.6, label="MED (>0.2)")
    for bar, v in zip(bars, [ks_vals[i] for i in sorted_idx]):
        ax.text(v + 0.01, bar.get_y() + bar.get_height()/2,
                f"{v:.3f}", va="center", fontsize=8.5)
    ax.set_xlim(0, 1.15)
    ax.set_xlabel("KS statistic")
    ax.set_title(f"Source -> {target_label}", fontweight="bold", color=col)
    ax.legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.show()

# ── CDF visualisation for the most-shifted feature ───────────────────────────
top_feat_idx = int(np.argmax(ks_stat_wf))
feat_name    = FEAT_COLS[top_feat_idx]
feat_nice    = FEAT_NICE[top_feat_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"CDF visualisation — '{feat_nice}' (highest WF KS = {ks_stat_wf[top_feat_idx]:.3f})",
             fontweight="bold")

for ax, X_tgt, tgt_lbl, tgt_col, ks_v in [
    (axes[0], X_wf, "Whole Foods",  STORE_COLORS["whole_foods"],  ks_stat_wf[top_feat_idx]),
    (axes[1], X_th, "Thrift Store", STORE_COLORS["thrift_store"], ks_stat_th[top_feat_idx]),
]:
    for vals, lbl, col, ls in [
        (X_src[:,top_feat_idx], "Source", SOURCE_COLOR, "-"),
        (X_tgt[:,top_feat_idx], tgt_lbl, tgt_col,      "--"),
    ]:
        xs   = np.sort(vals)
        ys   = np.arange(1, len(xs)+1) / len(xs)
        ax.step(xs, ys, color=col, linewidth=2, linestyle=ls, label=lbl, where="post")
    # Mark the KS gap with a vertical line at the point of maximum difference
    all_x = np.sort(np.concatenate([X_src[:,top_feat_idx], X_tgt[:,top_feat_idx]]))
    cdf_s = np.searchsorted(np.sort(X_src[:,top_feat_idx]), all_x) / len(X_src)
    cdf_t = np.searchsorted(np.sort(X_tgt[:,top_feat_idx]),   all_x) / len(X_tgt)
    max_i = np.argmax(np.abs(cdf_s - cdf_t))
    ax.annotate("", xy=(all_x[max_i], cdf_t[max_i]),
                xytext=(all_x[max_i], cdf_s[max_i]),
                arrowprops=dict(arrowstyle="<->", color="crimson", lw=2))
    ax.text(all_x[max_i] + 0.05, (cdf_s[max_i]+cdf_t[max_i])/2,
            f"KS = {ks_v:.3f}", color="crimson", fontsize=9)
    ax.set_xlabel(f"{feat_nice} (standardised)"); ax.set_ylabel("CDF")
    ax.set_title(f"Source vs {tgt_lbl}", fontweight="bold", color=tgt_col)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 5. Method 2 — Wasserstein distance (Earth Mover's Distance)

**Formula:** W = ∫ |F_source(x) − F_target(x)| dx

Also called Earth Mover's Distance — the minimum "work" (mass × distance)
to transform one distribution into the other.

**Key difference from KS:** Wasserstein is sensitive to the *magnitude* of
the shift, not just whether there is a gap. A distribution uniformly shifted
by +2σ scores W≈2.0 but may have a small KS statistic if the shape is intact.

**Range:** 0 to ∞ (in std units on standardised data)  
**Detects:** Mean shift magnitude, overall spread changes  
**Threshold:** <0.4 LOW, 0.4–0.8 MED, >0.8 HIGH (on standardised data)

In [ ]:
wass_wf = [wasserstein_distance(X_src[:,i], X_wf[:,i]) for i in range(len(FEAT_COLS))]
wass_th = [wasserstein_distance(X_src[:,i], X_th[:,i]) for i in range(len(FEAT_COLS))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle("Method 2 — Wasserstein distance: source vs each target", fontweight="bold")

for ax, wass_vals, target_label, col in [
    (axes[0], wass_wf, "Whole Foods",  STORE_COLORS["whole_foods"]),
    (axes[1], wass_th, "Thrift Store", STORE_COLORS["thrift_store"]),
]:
    sorted_idx = np.argsort(wass_vals)[::-1]
    bar_colors = [col if v > 0.8 else ("#EF9F27" if v > 0.4 else "#B4B2A9")
                  for v in [wass_vals[i] for i in sorted_idx]]
    bars = ax.barh([FEAT_NICE[i] for i in sorted_idx],
                   [wass_vals[i] for i in sorted_idx],
                   color=bar_colors, edgecolor="white", height=0.65)
    ax.axvline(0.8, color="crimson",    linestyle="--", linewidth=1.2, alpha=0.7, label="HIGH (>0.8)")
    ax.axvline(0.4, color="darkorange", linestyle="--", linewidth=1.0, alpha=0.6, label="MED (>0.4)")
    for bar, v in zip(bars, [wass_vals[i] for i in sorted_idx]):
        ax.text(v + 0.02, bar.get_y() + bar.get_height()/2,
                f"{v:.2f}", va="center", fontsize=8.5)
    ax.set_xlabel("Wasserstein distance (std units)")
    ax.set_title(f"Source -> {target_label}", fontweight="bold", color=col)
    ax.legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.show()

# ── KS vs Wasserstein scatter — shows when they agree and disagree ────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("KS vs Wasserstein — when do they agree?", fontweight="bold")

for ax, ks_v, wass_v, tgt_lbl, col in [
    (axes[0], ks_stat_wf, wass_wf, "Whole Foods",  STORE_COLORS["whole_foods"]),
    (axes[1], ks_stat_th, wass_th, "Thrift Store", STORE_COLORS["thrift_store"]),
]:
    ax.scatter(ks_v, wass_v, c=[STORE_COLORS.get(FEAT_COLS[i].split("_")[0], col)
                                  for i in range(len(FEAT_COLS))],
               s=60, alpha=0.8, edgecolors="white", linewidths=0.5)
    for i, fn in enumerate(FEAT_NICE):
        if ks_v[i] > 0.6 or wass_v[i] > 1.5:
            ax.annotate(fn, (ks_v[i], wass_v[i]), textcoords="offset points",
                        xytext=(5, 2), fontsize=7.5, color="black")
    ax.set_xlabel("KS statistic"); ax.set_ylabel("Wasserstein distance")
    ax.set_title(f"Source -> {tgt_lbl}", fontweight="bold", color=col)

plt.tight_layout()
plt.show()

## 6. Method 3 — Mean shift and standard deviation ratio

The simplest and most interpretable method — captures the first two moments.

**Mean shift** = |mean_target − mean_source| (in std units, since data is standardised)  
**Std ratio**  = std_target / std_source

- Std ratio < 1 → target customers are more homogeneous (narrower spread)
- Std ratio > 1 → target customers are more diverse (wider spread)
- Std ratio = 1 → same spread, just shifted

**Strength:** Immediately human-readable, directional  
**Weakness:** Cannot detect bimodality or tail differences

In [ ]:
ms_wf, sr_wf, ms_th, sr_th = [], [], [], []
for i in range(len(FEAT_COLS)):
    ms_wf.append(abs(X_wf[:,i].mean() - X_src[:,i].mean()))
    sr_wf.append(X_wf[:,i].std()  / (X_src[:,i].std() + 1e-9))
    ms_th.append(abs(X_th[:,i].mean() - X_src[:,i].mean()))
    sr_th.append(X_th[:,i].std()  / (X_src[:,i].std() + 1e-9))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Method 3 — Mean shift and std ratio by feature", fontweight="bold")

for row, (ms_vals, sr_vals, tgt_lbl, col) in enumerate([
    (ms_wf, sr_wf, "Whole Foods",  STORE_COLORS["whole_foods"]),
    (ms_th, sr_th, "Thrift Store", STORE_COLORS["thrift_store"]),
]):
    # Mean shift
    ax = axes[row, 0]
    sorted_idx = np.argsort(ms_vals)[::-1]
    bar_colors = [col if v > 1.0 else ("#EF9F27" if v > 0.5 else "#B4B2A9")
                  for v in [ms_vals[i] for i in sorted_idx]]
    bars = ax.barh([FEAT_NICE[i] for i in sorted_idx],
                   [ms_vals[i]   for i in sorted_idx],
                   color=bar_colors, edgecolor="white", height=0.65)
    ax.axvline(1.0, color="crimson",    linestyle="--", linewidth=1.2, alpha=0.7, label="|shift|>1σ")
    ax.axvline(0.5, color="darkorange", linestyle="--", linewidth=1.0, alpha=0.6, label="|shift|>0.5σ")
    for bar, v in zip(bars, [ms_vals[i] for i in sorted_idx]):
        ax.text(v+0.02, bar.get_y()+bar.get_height()/2, f"{v:.2f}σ", va="center", fontsize=8.5)
    ax.set_xlabel("Mean shift (std units)")
    ax.set_title(f"Source -> {tgt_lbl}: mean shift", fontweight="bold", color=col)
    ax.legend(fontsize=8)

    # Std ratio
    ax = axes[row, 1]
    sorted_idx2 = np.argsort(np.abs(np.array(sr_vals) - 1))[::-1]
    bar_colors2 = [col if abs(v-1) > 0.3 else ("#EF9F27" if abs(v-1) > 0.15 else "#B4B2A9")
                   for v in [sr_vals[i] for i in sorted_idx2]]
    bars2 = ax.barh([FEAT_NICE[i] for i in sorted_idx2],
                    [sr_vals[i]   for i in sorted_idx2],
                    color=bar_colors2, edgecolor="white", height=0.65)
    ax.axvline(1.0, color="black", linestyle="-",  linewidth=1.5, alpha=0.5, label="No change (=1)")
    ax.axvline(1.3, color="crimson",    linestyle="--", linewidth=1.2, alpha=0.7)
    ax.axvline(0.7, color="crimson",    linestyle="--", linewidth=1.2, alpha=0.7, label="±30% change")
    for bar, v in zip(bars2, [sr_vals[i] for i in sorted_idx2]):
        ax.text(v+0.01, bar.get_y()+bar.get_height()/2, f"{v:.2f}x", va="center", fontsize=8.5)
    ax.set_xlabel("Std ratio (target / source)")
    ax.set_title(f"Source -> {tgt_lbl}: std ratio", fontweight="bold", color=col)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. Method 4 — Jensen-Shannon divergence

**Formula:** JS(P, Q) = 0.5 × KL(P||M) + 0.5 × KL(Q||M)  where M = (P+Q)/2

Jensen-Shannon divergence is a symmetric, bounded version of KL divergence.

**Why JS instead of KL?**
- KL is asymmetric: KL(P||Q) ≠ KL(Q||P)
- KL = ∞ when Q(x) = 0 but P(x) > 0 (no smoothing needed with JS)
- JS is bounded in [0, 1] — easy to interpret and compare across features

**Implementation:** We approximate P and Q using 30-bin histograms,
add a small smoothing constant (1e-10), then apply the JS formula.

**Range:** 0 (identical) to 1 (completely different)

In [ ]:
def js_divergence(a, b, n_bins=30):
    lo  = min(a.min(), b.min())
    hi  = max(a.max(), b.max())
    bins = np.linspace(lo, hi, n_bins + 1)
    bw   = bins[1] - bins[0]
    p, _ = np.histogram(a, bins=bins, density=True); p = p * bw + 1e-10; p /= p.sum()
    q, _ = np.histogram(b, bins=bins, density=True); q = q * bw + 1e-10; q /= q.sum()
    m = 0.5 * (p + q)
    return float(0.5 * entropy(p, m) + 0.5 * entropy(q, m))

js_wf = [js_divergence(X_src[:,i], X_wf[:,i]) for i in range(len(FEAT_COLS))]
js_th = [js_divergence(X_src[:,i], X_th[:,i]) for i in range(len(FEAT_COLS))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle("Method 4 — Jensen-Shannon divergence per feature", fontweight="bold")

for ax, js_vals, tgt_lbl, col in [
    (axes[0], js_wf, "Whole Foods",  STORE_COLORS["whole_foods"]),
    (axes[1], js_th, "Thrift Store", STORE_COLORS["thrift_store"]),
]:
    sorted_idx = np.argsort(js_vals)[::-1]
    bar_colors = [col if v > 0.3 else ("#EF9F27" if v > 0.1 else "#B4B2A9")
                  for v in [js_vals[i] for i in sorted_idx]]
    bars = ax.barh([FEAT_NICE[i] for i in sorted_idx],
                   [js_vals[i]   for i in sorted_idx],
                   color=bar_colors, edgecolor="white", height=0.65)
    ax.axvline(0.3, color="crimson",    linestyle="--", linewidth=1.2, alpha=0.7, label="HIGH (>0.3)")
    ax.axvline(0.1, color="darkorange", linestyle="--", linewidth=1.0, alpha=0.6, label="MED (>0.1)")
    for bar, v in zip(bars, [js_vals[i] for i in sorted_idx]):
        ax.text(v + 0.005, bar.get_y() + bar.get_height()/2,
                f"{v:.3f}", va="center", fontsize=8.5)
    ax.set_xlim(0, 0.85)
    ax.set_xlabel("JS divergence")
    ax.set_title(f"Source -> {tgt_lbl}", fontweight="bold", color=col)
    ax.legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.show()

## 8. Method 5 — Classifier discrepancy (most powerful)

**Idea:** Train a classifier to distinguish source from target samples.  
If it succeeds → distributions are different. If it performs at chance → same distribution.

**Why this is the most powerful method:**  
All previous methods are *univariate* — they look at each feature independently.  
Two distributions can look similar on every individual feature while being  
very different in their *joint* distribution (correlated features).  
The classifier captures ALL differences simultaneously, including interactions.

**Implementation:**
1. Label source samples = 0, target samples = 1  
2. Train GradientBoostingClassifier with 5-fold CV  
3. Normalized discrepancy = 2 × (accuracy − 0.5)  
   → 0.0 = indistinguishable, 1.0 = perfectly separated

**Also useful:** Feature importances show which features drive the separation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Method 5 - Classifier discrepancy: source vs each target", fontweight="bold")

for ax, X_tgt, tgt_lbl, col in [
    (axes[0], X_wf, "Whole Foods",  STORE_COLORS["whole_foods"]),
    (axes[1], X_th, "Thrift Store", STORE_COLORS["thrift_store"]),
]:
    X_clf = np.vstack([X_src, X_tgt])
    y_clf = np.array([0]*len(X_src) + [1]*len(X_tgt))

    clf = GradientBoostingClassifier(n_estimators=100, max_depth=3,
                                      learning_rate=0.1, random_state=42)
    acc  = cross_val_score(clf, X_clf, y_clf, cv=5, scoring="accuracy").mean()
    disc = 2 * (acc - 0.5)

    clf.fit(X_clf, y_clf)
    imp  = clf.feature_importances_
    sorted_idx = np.argsort(imp)

    bar_colors = [col if v > 0.08 else ("#EF9F27" if v > 0.04 else "#B4B2A9")
                  for v in imp[sorted_idx]]
    bars = ax.barh([FEAT_NICE[i] for i in sorted_idx],
                   imp[sorted_idx], color=bar_colors, edgecolor="white", height=0.65)
    for bar, v in zip(bars, imp[sorted_idx]):
        if v > 0.02:
            ax.text(v + 0.002, bar.get_y() + bar.get_height()/2,
                    f"{v:.3f}", va="center", fontsize=8.5)

    title = "Source -> {} | CV acc: {:.1%}  Disc: {:.3f}".format(tgt_lbl, acc, disc)
    ax.set_xlabel("Feature importance (source vs target classification)")
    ax.set_title(title, fontweight="bold", color=col)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  CV accuracy near 0.5 = stores are behaviorally indistinguishable")
print("  CV accuracy near 1.0 = stores are completely different in feature space")
print("  Our results: ~99.5% (WF) and 100% (Thrift) -> extreme separation")


## 9. Method 6 — PCA centroid distance

**Idea:** Project all stores into a shared PCA feature space.  
Measure the Euclidean distance between the mean positions (centroids) of source and target.

**Why PCA first?**  
Raw feature space has 13 correlated dimensions.  
PCA decorrelates and removes redundancy, giving a more  
meaningful distance that isn't inflated by correlated features.

**Interpretation:** Distance in standardised units.  
<1.5 = small shift, 1.5–3.0 = medium, >3.0 = large shift

In [ ]:
pca = PCA(n_components=6, random_state=42)
X_all = scaler.transform(df[FEAT_COLS].values.astype(float))
pca.fit(X_all)

Xp_src = pca.transform(X_src)
Xp_wf  = pca.transform(X_wf)
Xp_th  = pca.transform(X_th)

src_cen = Xp_src.mean(axis=0)
wf_cen  = Xp_wf.mean(axis=0)
th_cen  = Xp_th.mean(axis=0)

dist_wf = np.linalg.norm(src_cen - wf_cen)
dist_th = np.linalg.norm(src_cen - th_cen)

# ── PCA scatter ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Method 6 — PCA centroid distance", fontweight="bold")

ev = pca.explained_variance_ratio_

# Left: all stores
ax = axes[0]
groups_pca = [
    ("Source (Kroger+Sfw+Wmt)", X_src, SOURCE_COLOR),
    ("Whole Foods",             X_wf,  STORE_COLORS["whole_foods"]),
    ("Thrift Store",            X_th,  STORE_COLORS["thrift_store"]),
]
for label, Xs, col in groups_pca:
    Xp = pca.transform(Xs)
    ax.scatter(Xp[:,0], Xp[:,1], c=col, s=12, alpha=0.4, edgecolors="none", label=label)

# Centroids
for cen, col, lbl in [
    (src_cen, SOURCE_COLOR,                       "Source centroid"),
    (wf_cen,  STORE_COLORS["whole_foods"],         "WF centroid"),
    (th_cen,  STORE_COLORS["thrift_store"],        "Thrift centroid"),
]:
    ax.scatter(cen[0], cen[1], c=col, s=180, marker="*",
               edgecolors="white", linewidths=0.8, zorder=5)

# Draw arrows from source centroid to target centroids
for cen, col, d, lbl in [
    (wf_cen, STORE_COLORS["whole_foods"],  dist_wf, f"WF dist={dist_wf:.1f}σ"),
    (th_cen, STORE_COLORS["thrift_store"], dist_th, f"TH dist={dist_th:.1f}σ"),
]:
    ax.annotate("", xy=cen[:2], xytext=src_cen[:2],
                arrowprops=dict(arrowstyle="->", color=col, lw=2))
    mid = (src_cen[:2] + cen[:2]) / 2
    ax.text(mid[0]+0.1, mid[1]+0.1, lbl, color=col, fontsize=9, fontweight="bold")

ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}% var)"); ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}% var)")
ax.set_title("PC1 vs PC2 — centroids and shift arrows", fontweight="bold")
ax.legend(fontsize=8, markerscale=2)

# Right: explained variance
ax = axes[1]
ax.bar(range(1, 7), ev*100, color="#185FA5", edgecolor="white", alpha=0.8)
ax.plot(range(1, 7), np.cumsum(ev)*100, "o-", color="#A32D2D", linewidth=2, label="Cumulative %")
ax.set_xlabel("Principal component"); ax.set_ylabel("Explained variance (%)")
ax.set_title("PCA explained variance", fontweight="bold")
ax.legend(fontsize=9)
for i, v in enumerate(ev*100):
    ax.text(i+1, v+0.5, f"{v:.1f}%", ha="center", fontsize=8.5)

plt.tight_layout()
plt.show()

# Shift direction: which features contribute most to the centroid displacement?
shift_wf = wf_cen - src_cen
shift_th = th_cen - src_cen
contrib_wf = pca.components_.T @ shift_wf
contrib_th = pca.components_.T @ shift_th

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
fig.suptitle("Which features drive the centroid shift?", fontweight="bold")
for ax, contrib, tgt_lbl, col in [
    (axes[0], contrib_wf, "Whole Foods",  STORE_COLORS["whole_foods"]),
    (axes[1], contrib_th, "Thrift Store", STORE_COLORS["thrift_store"]),
]:
    sorted_idx = np.argsort(np.abs(contrib))[::-1]
    bar_cols = [col if contrib[i] > 0 else "#A32D2D" for i in sorted_idx]
    bars = ax.barh([FEAT_NICE[i] for i in sorted_idx],
                   [contrib[i]   for i in sorted_idx],
                   color=bar_cols, alpha=0.8, edgecolor="white")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Contribution to centroid shift (positive = target higher)")
    ax.set_title(f"Shift direction: Source -> {tgt_lbl}", fontweight="bold", color=col)

plt.tight_layout()
plt.show()

## 10. Composite shift score and decision rule

Combine all six methods into a single actionable score, then map to an alpha recommendation.

**Normalisation:**
- KS: already [0, 1]  
- Wasserstein: divide by 2 (practical max ~5 on standardised data)  
- JS: multiply by 3 (rescale to [0, 1] sensitivity range)  
- Mean shift: divide by 2  
- Classifier discrepancy: already [0, 1]  
- PCA centroid distance: divide by 12

**Decision rule:**
| Composite score | Shift level | Alpha recommendation |
|---|---|---|
| > 0.5 | LARGE | alpha ≤ 0.3 |
| 0.3 – 0.5 | MEDIUM | alpha ≤ 0.5 |
| ≤ 0.3 | SMALL | alpha ≤ 0.7 |

In [ ]:
# Re-run classifier disc for clean values
disc_results = {}
for X_tgt, tgt_key, tgt_lbl in [
    (X_wf, "whole_foods", "Whole Foods"),
    (X_th, "thrift_store", "Thrift Store"),
]:
    X_clf = np.vstack([X_src, X_tgt])
    y_clf = np.array([0]*len(X_src) + [1]*len(X_tgt))
    clf   = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
    acc   = cross_val_score(clf, X_clf, y_clf, cv=5, scoring="accuracy").mean()
    disc_results[tgt_key] = 2 * (acc - 0.5)

summary = []
for X_tgt, tgt_key, tgt_lbl in [
    (X_wf, "whole_foods", "Whole Foods"),
    (X_th, "thrift_store", "Thrift Store"),
]:
    avg_ks    = np.mean([ks_2samp(X_src[:,i], X_tgt[:,i]).statistic for i in range(len(FEAT_COLS))])
    avg_wass  = np.mean([wasserstein_distance(X_src[:,i], X_tgt[:,i]) for i in range(len(FEAT_COLS))])
    avg_js    = np.mean([js_divergence(X_src[:,i], X_tgt[:,i]) for i in range(len(FEAT_COLS))])
    avg_ms    = np.mean([abs(X_tgt[:,i].mean() - X_src[:,i].mean()) for i in range(len(FEAT_COLS))])
    clf_disc  = disc_results[tgt_key]
    pca_dist  = np.linalg.norm(pca.transform(X_tgt).mean(axis=0) - pca.transform(X_src).mean(axis=0))

    norm_ks   = avg_ks
    norm_wass = min(avg_wass / 2, 1.0)
    norm_js   = min(avg_js * 3,   1.0)
    norm_ms   = min(avg_ms / 2,   1.0)
    norm_disc = clf_disc
    norm_pca  = min(pca_dist / 12, 1.0)

    composite = (norm_ks + norm_wass + norm_js + norm_ms + norm_disc + norm_pca) / 6
    # note: previous sessions used 5-method average; here we use 6

    level = "LARGE" if composite > 0.5 else ("MEDIUM" if composite > 0.3 else "SMALL")
    alpha_rec = "≤ 0.3" if composite > 0.5 else ("≤ 0.5" if composite > 0.3 else "≤ 0.7")

    summary.append({
        "Target": tgt_lbl,
        "Avg KS":        round(avg_ks,   4),
        "Avg Wasserstein": round(avg_wass, 4),
        "Avg JS div.":   round(avg_js,   4),
        "Avg mean shift":round(avg_ms,   4),
        "Classifier disc.": round(clf_disc, 4),
        "PCA dist":      round(pca_dist, 4),
        "Composite":     round(composite,4),
        "Level":         level,
        "Alpha rec.":    alpha_rec,
    })

display(pd.DataFrame(summary).set_index("Target"))

# ── Spider / radar of composite method contributions ─────────────────────────
METH_LABELS = ["KS\n(avg)", "Wasserstein\n(÷2)", "JS div.\n(×3)",
               "Mean shift\n(÷2)", "Classifier\ndisc.", "PCA dist\n(÷12)"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5),
                          subplot_kw=dict(polar=True))
fig.suptitle("Composite shift score — method contributions per target", fontweight="bold")

for ax, entry in zip(axes, summary):
    vals = [entry["Avg KS"],
            min(entry["Avg Wasserstein"]/2, 1),
            min(entry["Avg JS div."]*3, 1),
            min(entry["Avg mean shift"]/2, 1),
            entry["Classifier disc."],
            min(entry["PCA dist"]/12, 1)]
    vals += vals[:1]
    col  = STORE_COLORS["whole_foods"] if "Whole" in entry["Target"] else STORE_COLORS["thrift_store"]
    angles = np.linspace(0, 2*np.pi, len(METH_LABELS), endpoint=False).tolist()
    angles += angles[:1]
    ax.plot(angles, vals, color=col, linewidth=2)
    ax.fill(angles, vals, color=col, alpha=0.2)
    ax.axhline(y=0.5, color="crimson", linestyle="--", linewidth=1, alpha=0.6)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(METH_LABELS, fontsize=8.5)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0]); ax.set_yticklabels(["","0.5","","1.0"], fontsize=7)
    ax.set_title(f"{entry['Target']}\nComposite={entry['Composite']:.3f} [{entry['Level']}]\nAlpha {entry['Alpha rec.']}",
                 fontsize=10, fontweight="bold", color=col, pad=20)

plt.tight_layout()
plt.show()

## 11. All six methods side by side — comparison heatmap

In [ ]:
# Build a feature × method matrix for each target
method_names = ["KS", "Wasserstein", "Mean shift", "JS div.", "Importance"]

# Per-feature scores (methods 1–4 + classifier importance)
clf_imp = {}
for X_tgt, tgt_key in [(X_wf,"whole_foods"),(X_th,"thrift_store")]:
    X_clf = np.vstack([X_src, X_tgt])
    y_clf = np.array([0]*len(X_src)+[1]*len(X_tgt))
    clf   = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
    clf.fit(X_clf, y_clf)
    clf_imp[tgt_key] = clf.feature_importances_

for X_tgt, tgt_key, tgt_lbl, col in [
    (X_wf, "whole_foods", "Whole Foods",  STORE_COLORS["whole_foods"]),
    (X_th, "thrift_store","Thrift Store", STORE_COLORS["thrift_store"]),
]:
    ks_v   = [ks_2samp(X_src[:,i], X_tgt[:,i]).statistic for i in range(len(FEAT_COLS))]
    wass_v = [wasserstein_distance(X_src[:,i], X_tgt[:,i]) for i in range(len(FEAT_COLS))]
    ms_v   = [abs(X_tgt[:,i].mean() - X_src[:,i].mean()) for i in range(len(FEAT_COLS))]
    js_v   = [js_divergence(X_src[:,i], X_tgt[:,i]) for i in range(len(FEAT_COLS))]
    imp_v  = clf_imp[tgt_key]

    # Normalise each method column to [0,1] for the heatmap
    def norm01(v):
        v = np.array(v, dtype=float)
        return (v - v.min()) / (v.max() - v.min() + 1e-12)

    mat = np.column_stack([norm01(ks_v), norm01(wass_v), norm01(ms_v),
                           norm01(js_v), norm01(imp_v)])

    fig, ax = plt.subplots(figsize=(9, 5.5))
    im = ax.imshow(mat, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(method_names))); ax.set_xticklabels(method_names, fontsize=10)
    ax.set_yticks(range(len(FEAT_COLS)));    ax.set_yticklabels(FEAT_NICE, fontsize=9)
    for r in range(len(FEAT_COLS)):
        for c in range(len(method_names)):
            ax.text(c, r, f"{mat[r,c]:.2f}", ha="center", va="center",
                    fontsize=7.5, color="white" if mat[r,c] > 0.65 else "black")
    plt.colorbar(im, ax=ax, label="Normalised shift score (0=low, 1=high)", shrink=0.75)
    ax.set_title(f"Per-feature shift heatmap — Source -> {tgt_lbl}\n"
                 f"(each method column normalised to [0,1])",
                 fontweight="bold", color=col, pad=10)
    plt.tight_layout(); plt.show()

## 12. Consequences for the ML pipeline

Large feature shift has five concrete consequences for income prediction.

In [ ]:
from sklearn.linear_model import Ridge
from scipy.stats import rankdata, spearmanr

def quick_rank_model(X_src, X_tgt, y_src, lo, hi, alphas):
    yr = (rankdata(y_src)-1)/(len(y_src)-1)
    rng = np.random.RandomState(42); bX,by=[],[]
    for a in alphas:
        for _ in range(100):
            i,j=rng.randint(len(X_src)),rng.randint(len(X_tgt))
            bX.append((1-a)*X_src[i]+a*X_tgt[j]); by.append((1-a)*yr[i]+a*0.5)
    m = Ridge(alpha=10.0)
    m.fit(np.vstack([X_src,bX]), np.concatenate([yr,by]))
    return lo + np.clip(m.predict(X_tgt),0,1)*(hi-lo)

# Retrieve actual income for evaluation
y_src_all = df[df["store"].isin(SOURCE_STORES)]["income_usd"].values
y_wf      = wf_df["income_usd"].values
y_th      = th_df["income_usd"].values

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Effect of feature shift on prediction — Spearman r and MAE", fontweight="bold")

alpha_sets = [[0.1,0.3,0.5,0.7],[0.3,0.5,0.7,0.9],[0.7,0.9],[0.9]]
alpha_labs = ["[0.1,0.3,0.5,0.7]\n(best)", "[0.3,0.5,0.7,0.9]\n(baseline)",
              "[0.7,0.9]", "[0.9 only]\n(worst)"]

for col_i, (alphas, alph_lab) in enumerate(zip(alpha_sets, alpha_labs)):
    for row_i, (X_tgt, y_tgt, lo, hi, tgt_lbl, tgt_col) in enumerate([
        (X_wf, y_wf, 90000,150000, "Whole Foods",  STORE_COLORS["whole_foods"]),
        (X_th, y_th, 10000, 50000, "Thrift Store", STORE_COLORS["thrift_store"]),
    ]):
        ax = axes[row_i, col_i]
        preds = quick_rank_model(X_src, X_tgt, y_src_all, lo, hi, alphas)
        mae   = np.mean(np.abs(y_tgt - preds))
        sp, _ = spearmanr(y_tgt, preds)

        ax.scatter(y_tgt/1000, preds/1000, c=tgt_col, s=12, alpha=0.45,
                   edgecolors="none")
        mn = min(y_tgt.min(), preds.min())/1000
        mx = max(y_tgt.max(), preds.max())/1000
        ax.plot([mn,mx],[mn,mx], "k--", linewidth=1, alpha=0.5)

        ax.set_title(f"{tgt_lbl}\n{alph_lab}\nMAE=${mae:,.0f}  r={sp:.3f}",
                     fontsize=9, fontweight="bold", color=tgt_col)
        if row_i == 1: ax.set_xlabel("Actual income ($k)")
        if col_i == 0: ax.set_ylabel("Predicted income ($k)")

plt.tight_layout()
plt.show()

print("\nKey observation:")
print("  Spearman r (rank correlation) is robust across alpha choices.")
print("  MAE is sensitive to alpha — higher alpha -> label collapse -> worse calibration.")
print("  Feature shift determines which alpha range is safe.")